主要实现功能：根据历史销售数据和月度系数生成预测数据

In [1]:
import pantab as pt
import pandas as pd
from datetime import datetime, timedelta

pd.options.mode.copy_on_write = True

In [2]:
hfile = f'../src_data/hyper/退货率带to_25-12-21_tag_size_review.hyper'

parm_month_path = '../src_data/月度系数/月度系数.xlsx'
sales_program_path = '../src_data/Listing销占比程序表-模板.xlsx'

#+ 1、构建带本月系数的销占比数据

In [3]:
# 读取“Listing销占比程序表-模板”表
sales_program = pd.read_excel(sales_program_path)

# 读取“月度系数”，parse_dates参数用于指定列被解析为日期时间类型。
parm_month = pd.read_excel(parm_month_path, parse_dates=['月份'])
parm_month['年月'] = parm_month['月份'].dt.to_period('M').astype(str)

# 获取当前月份的第一天
current_month_start = datetime(datetime.now().year, datetime.now().month, 1)

# 筛选出日期大于等于当前月份的数据 (也就是获取当前月份及未来月份的月度系数 数据)
parm_month = parm_month[parm_month['月份'] >= current_month_start]
parm_month = parm_month.sort_values(by=['Listing', '月份'], ascending=[True, True])

# 根据'店铺-站点', 'Listing'去除重复行，最终只保留当前月份的月度系数 数据
temp_parm = parm_month.drop_duplicates(subset=['店铺-站点', 'Listing'])

# 构建出带本月系数的销占比数据（将月度系数与销占比程序表合并，销占比程序表为主表）
prepaer_df = pd.merge(left=sales_program, right=temp_parm, how='left', on=['店铺-站点', 'Listing', '站点'])

In [4]:
# parm_month.to_excel("../file/parm_month2.xlsx", index = False)

In [5]:
parm_month = parm_month.sort_values(by=['Listing', '月份'], ascending=[True, True])

In [6]:
parm_month

,店铺-站点,站点,Listing,月份,系数,年月
49,SEEKWAY:US,US,KP001,2025-12-01,1.300000,2025-12
105,SEEKWAY:US,US,KP001,2026-01-01,0.600000,2026-01
106,SEEKWAY:US,US,KP001,2026-02-01,1.300000,2026-02
107,SEEKWAY:US,US,KP001,2026-03-01,2.300000,2026-03
108,SEEKWAY:US,US,KP001,2026-04-01,1.100000,2026-04
...,...,...,...,...,...,...
438,RIVMOUNT:US,US,WP001,2026-02-01,0.926783,2026-02
439,RIVMOUNT:US,US,WP001,2026-03-01,1.342125,2026-03
440,RIVMOUNT:US,US,WP001,2026-04-01,1.149055,2026-04
441,RIVMOUNT:US,US,WP001,2026-05-01,1.496241,2026-05


In [7]:
parm_month['年月'].head(1)

49    2025-12
Name: 年月, dtype: object

In [8]:
temp_parm

,店铺-站点,站点,Listing,月份,系数,年月
49,SEEKWAY:US,US,KP001,2025-12-01,1.300000,2025-12
68,SEEKWAY:US,US,KSP001,2025-12-01,1.300000,2025-12
140,rivbos:US,US,RB831,2025-12-01,1.410200,2025-12
368,rivbos:US,US,RB832,2025-12-01,1.410200,2025-12
170,rivbos:US,US,RB833,2025-12-01,1.410200,2025-12
200,rivbos:US,US,RBK002,2025-12-01,1.608300,2025-12
230,rivbos:US,US,RBK003,2025-12-01,1.608300,2025-12
260,rivbos:US,US,RBK004,2025-12-01,1.608300,2025-12
290,rivbos:US,US,RBK023,2025-12-01,2.231900,2025-12
386,rivbos:US,US,RBK0231,2025-12-01,1.608300,2025-12


In [9]:
current_month_start

datetime.datetime(2025, 12, 1, 0, 0)

In [10]:
print("销占比程序表总行数：", len(sales_program), '\n月度系数表总行数：', len(temp_parm), '\n合并后总行数：', len(prepaer_df))

销占比程序表总行数： 3259 
月度系数表总行数： 24 
合并后总行数： 3259


In [11]:
prepaer_df.head()

,站点,负责人,Listing,款式,MSKU,积加SKU,规模定位,款式销占比,SKU销占比,需求定位,...,FNSKU,仓储类型,一级品类,品类A,品类B,定位-集合,需求定位-集合,月份,系数,年月
0,US,Ash,SK001,701条纹黑,SK001-701 Stripe Black 44-45,SK001-701 Stripe Black 44-45,长尾,0.0,0.1399,废弃,...,X0026DDW2N,鞋靴,SE_Tshoes-SEEKWAY薄底水鞋,水鞋,薄底水鞋,CA：短尾 US：长尾 US：短尾,CA：维护 US：废弃 US：维护,2025-12-01,1.084672,2025-12
1,US,Ash,SK001,701条纹黑,SK001-701 Stripe Black 42-43,SK001-701 Stripe Black 42-43,长尾,0.0,0.2140,废弃,...,X0026AMBR3,鞋靴,SE_Tshoes-SEEKWAY薄底水鞋,水鞋,薄底水鞋,CA：短尾 US：长尾 US：短尾,CA：维护 US：废弃 US：维护,2025-12-01,1.084672,2025-12
2,US,Ash,SK001,701条纹黑,SK001-701 Stripe Black 40-41,SK001-701 Stripe Black 40-41,长尾,0.0,0.2906,废弃,...,X0026AMBQJ,鞋靴,SE_Tshoes-SEEKWAY薄底水鞋,水鞋,薄底水鞋,CA：短尾 US：长尾 US：短尾,CA：维护 US：废弃 US：维护,2025-12-01,1.084672,2025-12
3,US,Ash,SK001,701条纹黑,SK001-701 Stripe Black 46-47,SK001-701 Stripe Black 46-47,长尾,0.0,0.0899,废弃,...,X0026FVERV,鞋靴,SE_Tshoes-SEEKWAY薄底水鞋,水鞋,薄底水鞋,CA：短尾 US：长尾 US：短尾,CA：维护 US：废弃 US：维护,2025-12-01,1.084672,2025-12
4,US,Ash,SK001,701条纹黑,SK001-701 Stripe Black 38-39,SK001-701 Stripe Black 38-39,长尾,0.0,0.2185,废弃,...,X0026CJQQ5,鞋靴,SE_Tshoes-SEEKWAY薄底水鞋,水鞋,薄底水鞋,CA：短尾 US：长尾 US：短尾,CA：维护 US：废弃 US：维护,2025-12-01,1.084672,2025-12


In [12]:
# prepaer_df.to_excel("../file/销占比与当月月度系数合并结果.xlsx", index = False)

# 2、处理历史销量

In [12]:
sales_df = pt.frame_from_hyper(hfile, table="return_review_data")

In [13]:
# 获取指定维度数据
sales_df = sales_df[['purchaseDate', '销量', 'Msku', '店铺-站点', 'SKU']]
sales_df.rename(columns={'Msku': 'MSKU'}, inplace=True)

# 删除空值
sales_df.dropna(subset=['销量'], inplace=True)

In [14]:
# 获取指定店铺数据加快处理速度
# TODO 此处可以修改
sales_df = sales_df[sales_df['店铺-站点'].isin(['CIKERWEL:US','SEEKWAY:US', 'rivbos:US', 'RIVMOUNT:US','SEEKWAY:CA','RIVMOUNT:CA'])]

# 3、分别获取近N天销量和近15天销量

In [15]:
# TODO 输入参数
year = input('请输入年份：')
month = input('请输入月份：')
days = input('请输入日期：')
sales_df['now'] = f'{year}-{month}-{days}'

# 获取日期格式
sales_df['now'] = pd.to_datetime(sales_df['now'])
sales_df['年月'] =  sales_df['now'].dt.to_period('M').astype(str)
sales_df['purchaseDate'] = pd.to_datetime(sales_df['purchaseDate'])

请输入年份： 2025
请输入月份： 12
请输入日期： 23


In [16]:
def get_last_month_days():
    # 获取上月总天数
    now = datetime.now()
    # 获取当月第一天，然后减去一天就是上个月最后一天
    first_day_this_month = datetime(now.year, now.month, 1)
    last_day_last_month = first_day_this_month - timedelta(days=1)
    return last_day_last_month.day

# 使用示例
print(f"上月总天数: {get_last_month_days()}天")

上月总天数: 30天


In [17]:
# 销量数据：获取四个时间：订单购买时间与现在时间的差（天数）、本月总天数、上月总天数、本月度过的天数

# 当前时间到订单购买时间天数差
sales_df['days_diff'] = (sales_df['now'] - sales_df['purchaseDate']).dt.days
# 获取指定日期所在月天数.
# TODO 控制近期销量（获取本月的总天数）
sales_df['days_in_month'] = sales_df['now'].apply(lambda x: x.days_in_month)
# 获取上月总天数
sales_df['days_in_lastmonth'] = get_last_month_days()

# 获取本月度过的天数（假设今天是4月22日，那就是度过了21天）
sales_df['curr_day']= sales_df['now'].dt.day - 1

In [18]:
sales_df['days_diff'].head()

35    1061
36    1051
37    1050
38    1046
39    1006
Name: days_diff, dtype: int64

In [19]:
sales_df['days_in_month'].head()

35    31
36    31
37    31
38    31
39    31
Name: days_in_month, dtype: int64

In [20]:
sales_df['days_in_lastmonth'].head()

35    30
36    30
37    30
38    30
39    30
Name: days_in_lastmonth, dtype: int64

In [21]:
sales_df['curr_day'].head()

35    22
36    22
37    22
38    22
39    22
Name: curr_day, dtype: int32

In [22]:
sales_df.tail()

,purchaseDate,销量,MSKU,店铺-站点,SKU,now,年月,days_diff,days_in_month,days_in_lastmonth,curr_day
1333241,2024-07-07,1.0,SWR005-502 circular white 38-39,RIVMOUNT:CA,SWS005-502 circular white 38-39,2025-12-23,2025-12,534,31,30,22
1333242,2024-07-03,1.0,SWR005-502 circular white 36-37,RIVMOUNT:CA,SWS005-502 circular white 36-37,2025-12-23,2025-12,538,31,30,22
1333243,2024-07-01,2.0,SWR005-502 circular white 40-41,RIVMOUNT:CA,SWS005-502 circular white 40-41,2025-12-23,2025-12,540,31,30,22
1333244,2024-07-02,1.0,SWR005-502 circular white 40-41,RIVMOUNT:CA,SWS005-502 circular white 40-41,2025-12-23,2025-12,539,31,30,22
1333245,2024-07-03,1.0,SWR005-502 circular white 40-41,RIVMOUNT:CA,SWS005-502 circular white 40-41,2025-12-23,2025-12,538,31,30,22


## 3.1 获取近N天销量
## 3.2 获取近15天销量

In [22]:
# def calculate_sales_by_month(sales_df, separate_days):
#     """
#     计算最近N天内每个店铺每种商品的本月销量和上月销量。
    
#     :param sales_df: 销售数据的DataFrame
#     :param separate_days: 分割天数的阈值(15天/N天)
#     :return: 包含本月销量和上月销量的DataFrame
#     """
    
#     # 确保days_diff列没有异常值，这里假设days_diff是日期差，应为非负数
#     if not all(sales_df['days_diff'] >= 0):
#         raise ValueError("days_diff列包含负值或异常值")
    
#     # 分割天数过滤 (获取最近一个月或者最近15天的销售数据)
#     recently_n_sales = sales_df.loc[sales_df['days_diff'] <= separate_days]
#     # print("天数差：",recently_n_sales['days_diff'].unique())
#     # print("本月度过的天数：",recently_n_sales['curr_day'].unique())

#     # print(recently_n_sales.head())
    
#     # 边界条件检查
#     if recently_n_sales.empty:
#         raise ValueError("没有符合条件的销售数据")

#     # 进一步将 recently_n_sales 分为两部分
#     # 计算本月销量和上月销量
#     current_month_sales_n = recently_n_sales.loc[recently_n_sales['days_diff'] <= recently_n_sales['curr_day']]
#     # print("本月销量:")
#     # print("\t天数差：",current_month_sales_n['days_diff'].unique())
#     # print("\t本月度过的天数：",current_month_sales_n['curr_day'].unique())
    
#     last_month_sales_n = recently_n_sales.loc[recently_n_sales['days_diff'] > recently_n_sales['curr_day']]
#     # print("上月销量:")
#     # print("\t天数差：",last_month_sales_n['days_diff'].unique())
#     # print("\t本月度过的天数：",last_month_sales_n['curr_day'].unique())

#     print("\n")
#     # 使用函数简化重复逻辑
#     def pivot_sales(df, column_name):
#         return df.pivot_table(index=['店铺-站点', 'MSKU'], values='销量', aggfunc='sum', fill_value=0).reset_index().rename(columns={'销量': column_name})
    
#     if current_month_sales_n.empty:
#         current_month_sales_n = pd.DataFrame({'店铺-站点': [], 'MSKU': [], '本月销量': []})
#     else:
#         current_month_sales_n = pivot_sales(current_month_sales_n, '本月销量')
#     # print("本月销量\n", current_month_sales_n.head())
        
#     if last_month_sales_n.empty:
#         last_month_sales_n = pd.DataFrame({'店铺-站点': [], 'MSKU': [], '上月销量': []})
#     else:
#         last_month_sales_n = pivot_sales(last_month_sales_n, '上月销量')
#     # print("上月销量\n",last_month_sales_n.head())
        
#     # print("\n\n\n")
#     return current_month_sales_n, last_month_sales_n

In [23]:
sales_df[sales_df['days_diff'] < 0]

,purchaseDate,销量,MSKU,店铺-站点,SKU,now,年月,days_diff,days_in_month,days_in_lastmonth,curr_day


In [55]:
def calculate_sales_by_month(sales_df, separate_days):
    """
    计算最近N天内每个店铺每种商品的本月销量和上月销量。
    
    :param sales_df: 销售数据的DataFrame
    :param separate_days: 分割天数的阈值
    :return: 包含本月销量和上月销量的DataFrame
    """
    
    # 确保days_diff列没有异常值，这里假设days_diff是日期差，应为非负数
    if not all(sales_df['days_diff'] >= 0):
        raise ValueError("days_diff列包含负值或异常值")
    
    # 分割天数过滤 (获取最近一个月或者最近15天的销售数据) separate_days为30（28，29，30，31）天或15天
    recently_n_sales = sales_df.loc[sales_df['days_diff'] <= separate_days]  # 筛选出小于等于近n天的销售数据
    # print("天数差：",recently_n_sales['days_diff'].unique())
    # print("本月度过的天数：",recently_n_sales['curr_day'].unique())

    # print(recently_n_sales.head())
    
    # 边界条件检查
    if recently_n_sales.empty:
        raise ValueError("没有符合条件的销售数据")

    print(recently_n_sales.columns)
    # 进一步将 recently_n_sales 分为两部分
    # 计算本月销量和上月销量
    current_month_sales_n = recently_n_sales.loc[recently_n_sales['days_diff'] <= recently_n_sales['curr_day']] # 小于等于当前月份度过的天数（本月销量）
    last_month_sales_n = recently_n_sales.loc[recently_n_sales['days_diff'] > recently_n_sales['curr_day']]  # 大于当月度过的天数（上月销量）
    
    alast_month_sales_n = last_month_sales_n.loc[last_month_sales_n['days_diff'] > last_month_sales_n['days_in_lastmonth']+last_month_sales_n['curr_day']] # (上上月销量）
    month_sales_n = last_month_sales_n.loc[last_month_sales_n['days_diff'] <= last_month_sales_n['days_in_lastmonth']+last_month_sales_n['curr_day']] # (上上月销量）

    current_month_sales_n.to_excel("../file/current_month_sales_n60.xlsx", index = False)
    last_month_sales_n.to_excel("../file/last_month_sales_n60.xlsx", index = False)
    alast_month_sales_n.to_excel("../file/alast_month_sales_n60.xlsx", index = False)
    month_sales_n.to_excel("../file/month_sales_n60.xlsx", index = False)
    
    # # 使用函数简化重复逻辑
    # def pivot_sales(df, column_name):
    #     return df.pivot_table(index=['店铺-站点', 'MSKU'], values='销量', aggfunc='sum', fill_value=0).reset_index().rename(columns={'销量': column_name})
    
    # if current_month_sales_n.empty:
    #     current_month_sales_n = pd.DataFrame({'店铺-站点': [], 'MSKU': [], '本月销量': []})
    # else:
    #     current_month_sales_n = pivot_sales(current_month_sales_n, '本月销量')
        
    # if last_month_sales_n.empty:
    #     last_month_sales_n = pd.DataFrame({'店铺-站点': [], 'MSKU': [], '上月销量': []})
    # else:
    #     last_month_sales_n = pivot_sales(last_month_sales_n, '上月销量')
    
    # return current_month_sales_n, last_month_sales_n

In [56]:
# 获取本月总天数
# 计算最多近一个31天销量 days_in_month 取值区间为[28, 29, 30, 31]
days_in_month = sales_df.iloc[0]['days_in_month']   # 本月总天数。iloc[行，列]  iloc[:, :]
days_in_month_60 = sales_df.iloc[0]['days_in_month'] + sales_df.iloc[0]['days_in_lastmonth']
parms_day = [days_in_month_60, days_in_month, 15]
file_names = ['60', '30', '15']
# parms_day = [days_in_month, 15]
# file_names = ['30', '15']


for day, file_name in zip(parms_day, file_names):
    calculate_sales_by_month(sales_df=sales_df, separate_days=day)
    break

Index(['purchaseDate', '销量', 'MSKU', '店铺-站点', 'SKU', 'now', '年月', 'days_diff',
       'days_in_month', 'days_in_lastmonth', 'curr_day'],
      dtype='object')


In [27]:
# 获取本月总天数
# 计算最多近一个31天销量 days_in_month 取值区间为[28, 29, 30, 31]
days_in_month = sales_df.iloc[0]['days_in_month']   # 本月总天数。iloc[行，列]  iloc[:, :]
days_in_month_60 = sales_df.iloc[0]['days_in_month'] + sales_df.iloc[0]['days_in_lastmonth']
parms_day = [days_in_month_60, days_in_month, 15]
file_names = ['60', '30', '15']


for day, file_name in zip(parms_day, file_names):
    current_month_sales_n, last_month_sales_n = calculate_sales_by_month(sales_df=sales_df, separate_days=day)

    # 在ret_df_n中添加本月销量和上月销量
    ret_df_n = pd.merge(left=prepaer_df, right=current_month_sales_n, how='left', on=['店铺-站点', 'MSKU'])
    ret_df_n['本月销量'] = ret_df_n['本月销量'].fillna(0)

    ret_df_n = pd.merge(left=ret_df_n, right=last_month_sales_n, how='left', on=['店铺-站点', 'MSKU'])
    ret_df_n['上月销量'] = ret_df_n['上月销量'].fillna(0)

    # 计算MSKU预估销量
    month_parms = 2 if day == 15 else 1
    ret_df_n['预估销量'] = (ret_df_n['上月销量'] + ret_df_n['本月销量'] / ret_df_n['系数']) * ret_df_n['系数'] * month_parms
    # print(ret_df_n.columns)
    # ret_df_n.to_excel("../file/ret_df_n.xlsx", index = False)
    # break
    
    # 获取款式销量和Listing销量为计算销占比做准备
    style_sales_n = ret_df_n.pivot_table(index=['店铺-站点', 'Listing','款式'], values=['预估销量'], aggfunc=sum).rename(columns={'预估销量': '款式销量'})
    listing_sales_n = ret_df_n.pivot_table(index=['店铺-站点', 'Listing'], values=['预估销量'], aggfunc=sum).rename(columns={'预估销量': 'Listing销量'})

    # 匹配款式销量
    ret_zhanbi_parms = pd.merge(left=ret_df_n, right=style_sales_n, how='left', left_on=['店铺-站点', 'Listing','款式'], right_index=True)
    # 匹配Listing销量
    ret_zhanbi_parms = pd.merge(left=ret_zhanbi_parms, right=listing_sales_n, how='left', left_on=['店铺-站点', 'Listing'], right_index=True)

    # 计算销占比
    ret_zhanbi_parms['SKU销占比']= (ret_zhanbi_parms['预估销量'] / ret_zhanbi_parms['款式销量']).fillna(0)


    ret_zhanbi_parms['款式销占比']= (ret_zhanbi_parms['款式销量'] / ret_zhanbi_parms['Listing销量']).fillna(0)


    # 构建月度销量
    listing_start_sales = ret_df_n.pivot_table(index=['站点', 'Listing', '月份'], values=['预估销量'], aggfunc='sum')
    # print(listing_start_sales)
    # listing_start_sales.to_excel("../file/listing_start_sales1.xlsx", index = False)
    
    # 匹配月度系数
    listing_start_sales = pd.merge(left=parm_month, right=listing_start_sales, left_on=['站点', 'Listing', '月份'], right_index=True, how='left').reset_index()
    # print(listing_start_sales)
    # listing_start_sales.to_excel("../file/listing_start_sales2.xlsx", index = False)
    # print(len(listing_start_sales))

    # 这段代码就是一个**“向下填充”**的逻辑，不过它并不是简单地复制上一行的值，而是把上一行的“预估销量”乘以本行的一个“系数”，作为当前行的“预估销量”。
    for index in range(1, len(listing_start_sales)):
        if pd.isna(listing_start_sales.loc[index, '预估销量']):
            listing_start_sales.loc[index, '预估销量'] = listing_start_sales.loc[index - 1, '预估销量'] * listing_start_sales.loc[index, '系数']

    
    ret_zhanbi_parms.to_excel(f'../src_data/Listing销占比程序表-{year}{month}{days}-{file_name}.xlsx', index=False, columns=sales_program.columns)

    listing_start_sales.rename(columns={'预估销量': 'Listing_月度预估销量'}, inplace=True)
    listing_start_sales.to_excel(f'../src_data/Listing预估表-{year}{month}{days}-{file_name}.xlsx', index=False, columns=['站点', 'Listing', '月份', 'Listing_月度预估销量'])


Index(['站点', '负责人', 'Listing', '款式', 'MSKU', '积加SKU', '规模定位', '款式销占比',
       'SKU销占比', '需求定位', '发货定位', '备货定位', '店铺', '运营保底下单天数', '总发货天数', '快递', '空运',
       '海运', '店铺-站点', '品牌', 'FNSKU', '仓储类型', '一级品类', '品类A', '品类B', '定位-集合',
       '需求定位-集合', '月份', '系数', '年月', '本月销量', '上月销量', '预估销量'],
      dtype='object')


In [28]:
ret_df_n

,站点,负责人,Listing,款式,MSKU,积加SKU,规模定位,款式销占比,SKU销占比,需求定位,...,品类A,品类B,定位-集合,需求定位-集合,月份,系数,年月,本月销量,上月销量,预估销量
0,CA,Ash,SK001,731黑,SK001-731 Black 34-35,SK001-731 Black 34-35,短尾,0.000,0.000000,废弃,...,水鞋,薄底水鞋,CA：短尾 US：中尾,CA：废弃 US：维护,NaT,NaN,NaN,0.0,0.0,NaN
1,CA,Ash,SK001,731黑,SK001-731 Black 36-37,SK001-731 Black 36-37,短尾,0.000,0.057007,废弃,...,水鞋,薄底水鞋,CA：短尾 US：中尾,CA：废弃 US：维护,NaT,NaN,NaN,0.0,0.0,NaN
2,CA,Ash,SK001,731黑,SK001-731 Black 38-39,SK001-731 Black 38-39,短尾,0.000,0.243186,废弃,...,水鞋,薄底水鞋,CA：短尾 US：中尾,CA：废弃 US：维护,NaT,NaN,NaN,0.0,0.0,NaN
3,CA,Ash,SK001,731黑,SK001-731 Black 40-41,SK001-731 Black 40-41,短尾,0.000,0.253592,废弃,...,水鞋,薄底水鞋,CA：短尾 US：中尾,CA：废弃 US：维护,NaT,NaN,NaN,0.0,0.0,NaN
4,CA,Ash,SK001,731黑,SK001-731 Black 42-43,SK001-731 Black 42-43,短尾,0.000,0.222216,废弃,...,水鞋,薄底水鞋,CA：短尾 US：中尾,CA：废弃 US：维护,NaT,NaN,NaN,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3982,US,ZHENFEI,SP006,AE099全黑,SP006-AE099 All Black 42,SP006-AE099 All Black 42,长尾,0.022,0.140000,维护,...,水鞋,溯溪水鞋,US：长尾,US：维护,2025-12-01,0.94,2025-12,3.0,0.0,6.0
3983,US,ZHENFEI,SP006,AE099全黑,SP006-AE099 All Black 43,SP006-AE099 All Black 43,长尾,0.022,0.110000,维护,...,水鞋,溯溪水鞋,US：长尾,US：维护,2025-12-01,0.94,2025-12,2.0,0.0,4.0
3984,US,ZHENFEI,SP006,AE099全黑,SP006-AE099 All Black 44,SP006-AE099 All Black 44,长尾,0.022,0.120000,维护,...,水鞋,溯溪水鞋,US：长尾,US：维护,2025-12-01,0.94,2025-12,1.0,0.0,2.0
3985,US,ZHENFEI,SP006,AE099全黑,SP006-AE099 All Black 45,SP006-AE099 All Black 45,长尾,0.022,0.090000,维护,...,水鞋,溯溪水鞋,US：长尾,US：维护,2025-12-01,0.94,2025-12,2.0,0.0,4.0


In [29]:
# ret_df_n.to_excel("../file/generateParms-ret_df_n.xlsx", index = False)